# PHAN TICH DU LIEU TUYEN DUNG IT TAI VIET NAM

Notebook nay trinh bay quy trinh thu thap du lieu, thong ke mo ta, cac buoc tien xu ly du lieu va khung truc quan hoa cho bai toan du bao muc luong `salary_avg`.

## 1. Data and Setup

Du lieu duoc crawl tu hai nguon la `ITViec` va `TopCV`. Sau khi thu thap, du lieu raw se duoc merge, lam sach, chuan hoa salary, suy ra mot so bien category va trich xuat skill phuc vu EDA va modeling.

In [3]:
from pathlib import Path
import inspect
import pandas as pd
from IPython.display import Markdown, display

from src.processing.clean_jobs import (
    clean_jobs,
    compute_salary_fields,
    normalize_company_type,
    normalize_level,
    normalize_location,
    normalize_remote_option,
    normalize_text,
    parse_experience_years,
)
from src.processing.extract_skills import enrich_with_skill_features, extract_skills_from_row

BASE_DIR = Path.cwd()
RAW_DIR = BASE_DIR / 'data' / 'raw'
CLEAN_DIR = BASE_DIR / 'data' / 'clean-data'

raw_itviec_path = RAW_DIR / 'itviec_jobs_20260315_115039.csv'
raw_topcv_path = RAW_DIR / 'topcv_jobs_20260314_231436.csv'
clean_path = CLEAN_DIR / 'jobs_cleaned.csv'
features_path = CLEAN_DIR / 'jobs_features.csv'

df_itviec = pd.read_csv(raw_itviec_path)
df_topcv = pd.read_csv(raw_topcv_path)
df_clean = pd.read_csv(clean_path)
df_features = pd.read_csv(features_path)


## 2. Trinh bay ma nguon crawl du lieu



## 3. Tong quan du lieu thu thap

Truoc khi cleaning, can kiem tra kich thuoc du lieu, schema va mot vai mau dai dien cua tung nguon.

In [10]:
summary_df = pd.DataFrame({
    'dataset': ['ITViec raw', 'TopCV raw'],
    'rows': [len(df_itviec), len(df_topcv)],
    'columns': [df_itviec.shape[1], df_topcv.shape[1]],
})
summary_df

,dataset,rows,columns
0,ITViec raw,200,9
1,TopCV raw,809,9


**Schema cua 2 file raw**

In [5]:
print('ITViec columns:', list(df_itviec.columns))
print('TopCV columns:', list(df_topcv.columns))

ITViec columns: ['job_title', 'tech_stack', 'experience_years', 'location', 'company_name', 'remote_option', 'salary_min', 'salary_max', 'currency']
TopCV columns: ['job_title', 'tech_stack', 'experience_years', 'location', 'company_name', 'remote_option', 'salary_min', 'salary_max', 'currency']


**Mẫu dữ liệu thô**

In [6]:
display(df_itviec.head(3))
display(df_topcv.head(3))

,job_title,tech_stack,experience_years,location,company_name,remote_option,salary_min,salary_max,currency
0,Big Data Admin (Kafka/Docker/Data Warehousing/...,"Big Data, Data Warehousing, Docker, MySQL, Kaf...",1 years,Ha Noi,Giao Hàng Tiết Kiệm,onsite,600.0,1200.0,USD
1,Security Engineer (Cloud & CI/CD),"Security, DevSecOps, AWS, Linux, SIEM, CI/CD",2 years,Ha Noi,VIETTEL CUSTOMER SERVICE,onsite,1000.0,2000.0,USD
2,"Senior Embedded QT Architect (C++, Linux, AI-F...","Embedded, Clean Architecture, AI, Linux, C++, Qt",5-8 years,Ha Noi,NTP-Tech,onsite,2000.0,2000.0,USD


,job_title,tech_stack,experience_years,location,company_name,remote_option,salary_min,salary_max,currency
0,Junior Fullstack Developer (NODE.JS + REACT) (...,"fullstack developer, it - phần mềm, nghỉ thứ.....",2.0,Hà Nội,CÔNG TY CỔ PHẦN GIẢI PHÁP CÔNG NGHỆ ZENIFY,onsite,25000000.0,25000000.0,VND
1,Chuyên Viên Phát Triển Phần Mềm,"software engineer, nghỉ thứ 7, 3 năm kinh ng.....",3.0,Hà Nội & Hồ Chí Minh (mới),CÔNG TY CỔ PHẦN TẬP ĐOÀN MIK GROUP VIỆT NAM,onsite,NaN,NaN,NaN
2,Junior – Senior Java,"backend developer, nghỉ thứ 7, tuổi 25 - 30, 2...",2.0,Hà Nội,CÔNG TY CỔ PHẦN ADVANCE TECH VIỆT NAM,onsite,600.0,2000.0,USD


## 4. Thong ke mo ta du lieu

Phan nay tap trung vao thong ke co ban cua bo du lieu sau cleaning de danh gia muc do day du va muc do phu hop cho bai toan du bao `salary_avg`.

**Thong tin tong quan cua bo du lieu clean**

In [11]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 780 entries, 0 to 779
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   job_id               780 non-null    str    
 1   source               780 non-null    str    
 2   job_title            780 non-null    str    
 3   company_name         780 non-null    str    
 4   location             780 non-null    str    
 5   company_type         780 non-null    str    
 6   level                755 non-null    str    
 7   experience_years     709 non-null    float64
 8   salary_min           360 non-null    float64
 9   salary_max           360 non-null    float64
 10  salary_avg           360 non-null    float64
 11  salary_currency      360 non-null    str    
 12  remote_option        780 non-null    str    
 13  tech_stack           780 non-null    str    
 14  salary_is_estimated  780 non-null    int64  
 15  salary_min_original  360 non-null    float64
 16  s

**So luong tin theo nguon du lieu**

In [20]:
df_clean['source'].value_counts().to_frame('job_count')

,job_count
source,
topcv,581
itviec,199


**Top dia diem co nhieu tin tuyen dung nhat**

In [15]:
df_clean['location'].value_counts().head(10).to_frame('job_count')

,job_count
location,
Ha Noi,502
Hồ Chí Minh (Mới),115
Ho Chi Minh,104
Đà Nẵng (Mới),14
Da Nang,5
Cần Thơ (Mới),4
Hải Phòng (Mới),4
Others,3
Ho Chi Minh - Da Nang,3


**Top cap bac duoc tuyen nhieu nhat**

In [16]:
df_clean['level'].value_counts(dropna=False).to_frame('job_count')

,job_count
level,
Junior,293
Middle,207
Senior,149
Lead,61
Intern,36
NaN,25
Manager,9


**Thong ke mo ta cac bien dinh luong**

In [21]:
salary_cols = ['salary_min', 'salary_max', 'salary_avg', 'experience_years']
df_clean[salary_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
salary_min,360.0,2.310069e+07,1.373831e+07,1000000.0,14812500.0,20000000.0,30000000.0,90000000.0
salary_max,360.0,3.488542e+07,2.146899e+07,1000000.0,20000000.0,30000000.0,45000000.0,200000000.0
salary_avg,360.0,2.899306e+07,1.666915e+07,1000000.0,18000000.0,25250000.0,37500000.0,145000000.0
experience_years,709.0,2.950635e+00,1.720298e+00,1.0,2.0,3.0,4.0,15.0


**Ty le record co va khong co salary_avg**

In [17]:
salary_availability = pd.Series({
    'co_salary_avg': int(df_clean['salary_avg'].notna().sum()),
    'thieu_salary_avg': int(df_clean['salary_avg'].isna().sum()),
}).to_frame('count')
salary_availability['ratio'] = (salary_availability['count'] / len(df_clean)).round(4)
salary_availability

,count,ratio
co_salary_avg,360,0.4615
thieu_salary_avg,420,0.5385


## 5. Nhan xet ban dau tu thong ke mo ta

- Sau khi gộp 2 nguồn, bộ dữ liệu có thể dùng cho EDA và bài toán hồi quy dự báo `salary_avg`.
- Không phải mỗi tin tuyển dụng đều công khai lương, vì vậy `salary_avg` có tỷ lệ thiếu nhất định.
- Các cột `location`, `level`, `company_type`, `remote_option`, `tech_stack`  và nhóm skill là các đặc trưng có khả năng tác động đến salary.
- Việc xử lý missing salary cần được làm thận trọng: giữ lại record cho EDA nhưng loại khỏi tập train nếu không có target.

## 6. Tien xu ly du lieu

Phần này trình bày chi tiết từng bước tiền xử lý dữ liệu. Các logic được đóng gói gọn trong module `src/processing/`, tuân thủ rule: **không làm mất mát dữ liệu tùy tiện**, xử lý triệt để lỗi sinh ra từ crawler (nhập sai min/max, chuỗi text, thiếu loại tiền tệ).

### 6.1. Muc tieu cua qua trinh cleaning

Muc tieu cua cleaning trong de tai nay khong chi la xoa null hay bo trung lap. Quan trong hon, no can bien du lieu raw thanh mot bang nhat quan de co the:

- tao dung bien muc tieu `salary_avg`
- so sanh salary giua cac nguon tren cung mot don vi
- bien cac truong text thanh cac bien category co the phan tich
- giu lai nhung record van con gia tri cho EDA, ke ca khi thieu salary
- tao mot bo du lieu sach de lam nen cho feature engineering va modeling

### 6.2. Gop du lieu raw

Hai file raw CSV duoc doc va noi thanh mot bang duy nhat. Trong qua trinh nay, cot `source` duoc them vao de biet moi record den tu `itviec` hay `topcv`.

In [ ]:
df_raw_merged = pd.concat([
    df_itviec.assign(source='itviec'),
    df_topcv.assign(source='topcv')
], ignore_index=True)
df_raw_merged.head(3)

### 6.3. Lam sach text co ban

Cac cot text thuong co khoang trang thua, chuoi rong hoac bieu dien khong dong nhat. Ham `normalize_text()` duoc dung de:

- trim khoang trang dau va cuoi chuoi
- quy tat nhieu khoang trang lien tiep ve mot khoang trang
- bien chuoi rong thanh gia tri thieu (`NaN`)

Buoc nay duoc ap dung cho cac cot: `job_title`, `company_name`, `location`, `remote_option`, `tech_stack`.

In [ ]:
print(inspect.getsource(normalize_text))

sample_text = pd.Series(['  Ha Noi  ', 'Remote   only', '', None])
pd.DataFrame({
    'before': sample_text,
    'after': sample_text.apply(normalize_text)
})

### 6.4. Chuan hoa location va remote option

Dữ liệu raw có nhiều biến thể cho cùng một nội dung (ví dụ `Hà Nội`, `Ha Noi`, `HN`, `TP.HCM`, `Saigon`). Pipeline giải quyết bằng cách:

- Dùng `normalize_location()` đối chiếu với `LOCATION_MAP` mở rộng (hỗ trợ phân loại hơn 20 tỉnh thành phổ biến tại VN như Hà Nội, HCM, Đà Nẵng, Cần Thơ, Hải Phòng...).
- Xử lý các tin tuyển dụng đa địa điểm (multi-location): bóc tách từng địa điểm và nối lại bằng dấu `|`.
- Dùng `normalize_remote_option()` để thống nhất các format `on-site`, `WFH`, `work from home` về 3 chuẩn duy nhất: `onsite`, `remote`, `hybrid`.

In [ ]:
print(inspect.getsource(normalize_location))

location_demo = pd.DataFrame({
    'raw_location': ['H? N?i', 'Ha Noi', 'HCM', 'TP.HCM', 'H? N?i & H? Ch? Minh (m?i)'],
    'normalized_location': [
        normalize_location('H? N?i'),
        normalize_location('Ha Noi'),
        normalize_location('HCM'),
        normalize_location('TP.HCM'),
        normalize_location('H? N?i & H? Ch? Minh (m?i)')
    ]
})
location_demo

In [ ]:
remote_demo = pd.DataFrame({
    'raw_remote_option': ['on-site', 'onsite', 'office', 'WFH', 'work from home', 'hybrid'],
    'normalized_remote_option': [
        normalize_remote_option('on-site'),
        normalize_remote_option('onsite'),
        normalize_remote_option('office'),
        normalize_remote_option('WFH'),
        normalize_remote_option('work from home'),
        normalize_remote_option('hybrid')
    ]
})
remote_demo

### 6.5. Chuan hoa experience

Cột `experience_years` chứa text tự do như `1 years`, `Dưới 1 năm`, `2-5 years`. Hàm `parse_experience_years()` sử dụng Regex để tách số:

- Nếu có 1 số: lấy số đó làm số năm kinh nghiệm.
- Nếu có 1 khoảng (range, ví dụ `2-4 năm`): tính **trung bình (mean)** của khoảng để con số phản ánh đúng tính chất trung vị của yêu cầu thay vì thiên lệch về giá trị lớn nhất.

In [ ]:
print(inspect.getsource(parse_experience_years))

experience_demo = pd.DataFrame({
    'raw_experience': ['1 years', '2', '3+ years', 'At least 5 years', None],
    'parsed_experience': [
        parse_experience_years('1 years'),
        parse_experience_years('2'),
        parse_experience_years('3+ years'),
        parse_experience_years('At least 5 years'),
        parse_experience_years(None)
    ]
})
experience_demo

### 6.6. Xu ly salary va tao `salary_avg`

Đây là bước trọng tâm để tạo biến mục tiêu `Y` (`salary_avg`). Pipeline thực hiện:

1. **Chuyển đổi kiểu dữ liệu**: convert `salary_min` và `salary_max` về số thực.
2. **Xác định tiền tệ**: Dựa vào `currency` gốc hoặc suy luận default theo `source` (`itviec` mặc định là USD, `topcv` và `topdev` mặc định là VND).
3. **Quy đổi ngoại tệ**: Dùng `SALARY_EXCHANGE_RATES` quy tất cả về `VND`.
4. **Sửa lỗi data entry**: Nếu `salary_min > salary_max` (do người nhập liệu nhầm), tự động swap (đảo ngược) 2 cột.
5. **Xử lý thiếu khoảng thuật toán**: Nếu chỉ có 1 trong 2 đầu lương (chỉ có min hoặc chỉ có max), dùng chính đầu đó cho đầu còn lại và gắn cờ `salary_is_estimated = 1`.
6. **Tính trung bình**: `salary_avg = (salary_min + salary_max) / 2`.
7. **Lọc Outlier**: Xóa `salary_avg` nếu nằm ngoài biên độ thực tế của thị trường VN (dưới 1 triệu hoặc trên 500 triệu VND/tháng).

In [ ]:
print(inspect.getsource(compute_salary_fields))

In [ ]:
salary_demo = df_raw_merged[['source', 'salary_min', 'salary_max', 'currency']].head(8).copy()
salary_demo_before = salary_demo.copy()
salary_demo_after = compute_salary_fields(salary_demo)
pd.concat([
    salary_demo_before.add_prefix('before_'),
    salary_demo_after[['salary_min', 'salary_max', 'salary_avg', 'salary_currency', 'salary_is_estimated']].add_prefix('after_')
], axis=1)

### 6.7. Xu ly record thieu l??ng

Voi cac record khong co du lieu salary, pipeline khong loai bo ngay o buoc cleaning. Thay vao do:

- giu lai record trong `jobs_cleaned.csv` de van dung duoc cho EDA
- dat `salary_min`, `salary_max`, `salary_avg` la `NaN` neu khong co du lieu
- chi loai khoi tap train khi xay dung mo hinh hoi quy, vi khi do moi can target `Y = salary_avg`

Cach xu ly nay giup tranh tao du lieu gia va van bao ton thong tin thi truong tuyen dung.

In [ ]:
df_clean[df_clean['salary_avg'].isna()].head(5)

### 6.8. Tao bien category moi: `level` va `company_type`

Giúp chuyển hóa thông tin thô thành các features phân loại (categorical features) dọn đường cho Modeling:

- **`level` (Cấp bậc)**: Pipeline **ưu tiên sử dụng cột `level` gốc** từ raw data (nếu có). Nếu không có, thuật toán fallback sang nội suy dựa trên các keyword trong `job_title` (như `senior`, `lead`, `manager`) hoặc đối chiếu theo mốc `experience_years`.
- **`company_type` (Loại hình công ty)**: Mapping nâng cao. Thuật toán quét danh sách hơn 50 tập đoàn/công ty IT lớn tại VN (VNG, FPT, Tiki, CMC...) để nhận diện `Technology`. Sau đó dùng keyword matching để tra cứu các mảng `Finance`, `Healthcare`, `Education`, `Logistics`, `E-Commerce`.

In [ ]:
print(inspect.getsource(normalize_level))

level_demo = pd.DataFrame({
    'job_title': ['Junior Python Developer', 'Senior Backend Engineer', 'QA Manager', 'Data Engineer'],
    'experience_years': [1, 5, 6, 3],
    'level': [
        normalize_level('Junior Python Developer', 1),
        normalize_level('Senior Backend Engineer', 5),
        normalize_level('QA Manager', 6),
        normalize_level('Data Engineer', 3),
    ]
})
level_demo

In [ ]:
company_demo = pd.DataFrame({
    'company_name': [
        'Techcombank',
        'ABC Software Solutions',
        'Giao H?ng Ti?t Ki?m',
        'XYZ Academy',
        'Some Unknown Company'
    ]
})
company_demo['company_type'] = company_demo['company_name'].apply(normalize_company_type)
company_demo

### 6.9. Loai bo duplicate va tao `job_id`

Sau khi da chuan hoa du lieu, pipeline tao `job_id` tu cac truong nhan dien chinh va xoa cac record trung lap. Buoc nay giup tranh dem lap cung mot tin tuyen dung nhieu lan khi thong ke.

In [ ]:
raw_count = len(df_raw_merged)
clean_count = len(df_clean)
pd.DataFrame({
    'metric': ['raw_rows_after_merge', 'rows_after_cleaning', 'removed_rows'],
    'value': [raw_count, clean_count, raw_count - clean_count]
})

### 6.10. Feature engineering tu skill

Quy trình **NLP & Feature Engineering** trên text (`extract_skills.py`) sinh ra tập `jobs_features.csv`:

1. **Tổng hợp text**: Lấy dữ liệu từ `tech_stack`, `job_title`, và `job_description` (nếu có).
2. **Tokenizer & Normalizer**: Chuyển text về chữ thường, lược bỏ dấu tiếng Việt, loại bỏ ký tự lạ, và normalize các alias (ví dụ: `reactjs`, `react js` $\rightarrow$ `react`; `nodejs` $\rightarrow$ `node.js`).
3. **Lọc kỹ năng**: Đối chiếu với bộ từ điển `COMMON_SKILLS` (chứa hơn 60 công nghệ phổ biến) để lọc nhiễu.
4. **Tạo Label Binary**: Tạo loạt cột indicator `has_python`, `has_js_ts`, `has_sql`, `has_cloud`, `has_mobile`, `has_devops`... bằng giá trị nhị phân (0/1), cực kỳ phù hợp làm features để train model dự báo lương.

In [ ]:
print(inspect.getsource(extract_skills_from_row))

In [ ]:
feature_preview = enrich_with_skill_features(df_clean.head(5).copy())
feature_preview[['job_title', 'tech_stack', 'skills_extracted', 'skills_count', 'has_python', 'has_sql', 'has_cloud', 'has_devops']]

### 6.11. So sanh `jobs_cleaned.csv` va `jobs_features.csv`

Hai file dau ra phuc vu hai muc dich khac nhau:

- `jobs_cleaned.csv`: dung cho kiem tra chat luong du lieu va EDA co ban
- `jobs_features.csv`: dung cho cac buoc feature engineering va modeling

In [ ]:
comparison_df = pd.DataFrame({
    'dataset': ['jobs_cleaned.csv', 'jobs_features.csv'],
    'rows': [len(df_clean), len(df_features)],
    'columns': [df_clean.shape[1], df_features.shape[1]],
    'has_skills_extracted': ['skills_extracted' in df_clean.columns, 'skills_extracted' in df_features.columns],
    'has_skill_indicators': ['has_python' in df_clean.columns, 'has_python' in df_features.columns],
})
comparison_df

## 7. Truc quan hoa du lieu

Phan nay giu khung trinh bay cho cac bieu do, chua dien phan tich chi tiet.

### 7.1. Truc quan hoa don bien

- Phan bo `salary_avg` sau khi cleaning
- So luong tin theo `source`
- So luong tin theo `location`
- So luong tin theo `level`

In [ ]:
# TODO: Ve histogram/boxplot cho salary_avg
# TODO: Ve countplot cho source, location, level

### 7.2. Truc quan hoa hai bien va da bien

- `salary_avg` theo `level`
- `salary_avg` theo `location`
- `salary_avg` theo nhom skill
- So sanh salary giua `remote`, `hybrid`, `onsite`

In [ ]:
# TODO: Ve boxplot/barplot de so sanh salary theo level, location, remote_option
# TODO: Kiem tra moi quan he giua salary_avg va skills_count

### 7.3. Truc quan hoa nhieu chieu

- Heatmap tuong quan giua cac bien so
- Embedding hoac giam chieu cho nhom feature skill neu can
- Tong hop insight tu cac bieu do

In [ ]:
# TODO: Correlation heatmap
# TODO: TSNE/PCA neu can phan tich khong gian feature

## 8. Nhan xet va ket luan tam thoi

- Du lieu da duoc crawl tu 2 nguon va hop nhat thanh mot bang phuc vu bai toan du bao muc luong.
- Buoc cleaning giup chuan hoa salary, dia diem, kinh nghiem, suy ra category va loai bo duplicate.
- `jobs_cleaned.csv` giu du lieu da lam sach de phuc vu EDA.
- `jobs_features.csv` bo sung cac feature tu skill de san sang cho modeling.
- Cac phan truc quan hoa va dien giai insight chi tiet se duoc bo sung tiep theo.